# flit_core: 5 Basic Usage Examples

In [ ]:
# Example 1: Basic flit_core import and info
import flit_core

# Check basic module info
print(f"flit_core module: {flit_core.__name__}")
print(f"flit_core version: {flit_core.__version__}")

# Test buildapi module: the PEP 517 hooks flit_core exists to provide
import flit_core.buildapi
print(f"Build API module available: {flit_core.buildapi is not None}")
print(f"Build wheel function: {hasattr(flit_core.buildapi, 'build_wheel')}")
print(f"Build sdist function: {hasattr(flit_core.buildapi, 'build_sdist')}")
assert callable(flit_core.buildapi.build_wheel) and callable(flit_core.buildapi.build_sdist)

# Check config module
import flit_core.config
print(f"Config module available: {flit_core.config is not None}")
assert callable(flit_core.config.read_flit_config)


In [ ]:
# Example 2: Create minimal pyproject.toml
import tempfile
import os
from pathlib import Path

# Create temporary directory
temp_dir = Path(tempfile.mkdtemp())
print(f"Working in: {temp_dir}")

# Create minimal pyproject.toml content
pyproject_content = '''
[build-system]
requires = ["flit_core"]
build-backend = "flit_core.buildapi"

[project]
name = "test-package"
version = "0.1.0"
description = "Test package for flit_core"
authors = [{name = "Test Author", email = "test@example.com"}]
'''

pyproject_file = temp_dir / "pyproject.toml"
pyproject_file.write_text(pyproject_content)

# Create minimal module
module_file = temp_dir / "test_package.py"
module_file.write_text('"""Test module"""\n__version__ = "0.1.0"')

print(f"Project structure created successfully")


In [ ]:
# Example 3: Test flit_core config parsing
import flit_core.config
import tempfile
import os
from pathlib import Path

# Create test project structure
temp_dir = Path(tempfile.mkdtemp())
os.chdir(temp_dir)

# Create pyproject.toml
pyproject_content = '''
[build-system]
requires = ["flit_core"]
build-backend = "flit_core.buildapi"

[project]
name = "example-pkg"
version = "1.0.0"
description = "Example package"
'''

(temp_dir / "pyproject.toml").write_text(pyproject_content)
(temp_dir / "example_pkg.py").write_text('"""Example module"""')

# read_flit_config() takes the path of the pyproject.toml to read (flit_core >= 3.x signature: `(path)`)
config = flit_core.config.read_flit_config(temp_dir / "pyproject.toml")
print(f"Config loaded: {type(config)}")
print(f"Module name: {config.module}")
assert isinstance(config, flit_core.config.LoadedConfig), type(config)
assert config.module == "example_pkg", config.module
assert config.metadata["name"] == "example-pkg" and config.metadata["version"] == "1.0.0", config.metadata


In [ ]:
# Example 4: Test build API functions
import flit_core.buildapi
import tempfile

# Test get_requires_for_build functions
try:
    build_requires = flit_core.buildapi.get_requires_for_build_wheel()
    print(f"Wheel build requires: {build_requires}")
    
    sdist_requires = flit_core.buildapi.get_requires_for_build_sdist()
    print(f"Sdist build requires: {sdist_requires}")
    
    # Test prepare function
    metadata_dir = flit_core.buildapi.prepare_metadata_for_build_wheel(
        metadata_directory=tempfile.mkdtemp()
    )
    print(f"Metadata prepared: {metadata_dir is not None}")
    
except Exception as e:
    print(f"Build API test: {type(e).__name__}: {e}")
    raise AssertionError(
        f"flit_core example (cell 4) failed: {type(e).__name__}: {e}"
    ) from e

# Check buildapi module structure
buildapi_attrs = [attr for attr in dir(flit_core.buildapi) if not attr.startswith('_')]
print(f"Build API functions: {len(buildapi_attrs)}")
print(f"Available functions: {buildapi_attrs[:5]}...")  # First 5

In [ ]:
# Example 5: Test flit_core utilities
import flit_core.common
import flit_core.versionno

# Test version number parsing
try:
    version_info = flit_core.versionno.normalise_version("1.2.3")
    print(f"Normalized version: {version_info}")
    
    # Test invalid version handling
    try:
        invalid = flit_core.versionno.normalise_version("invalid")
    except flit_core.common.InvalidVersion as e:
        assert "PEP 440" in str(e), f"unexpected message: {e!r}"
        print(f"Invalid version rejected: {type(e).__name__}")
    else:
        raise AssertionError(f"normalise_version('invalid') was accepted: {invalid!r}")
        
except Exception as e:
    print(f"Version utilities: {type(e).__name__}: {e}")
    raise  # the enclosing catch-all used to swallow the narrowed check above; a failure must red the cell

# Check common utilities
common_attrs = [attr for attr in dir(flit_core.common) if not attr.startswith('_')]
print(f"Common utilities: {len(common_attrs)}")

# Test module inspection: Module locates an importable package on disk — flit_core itself here (measured: a package)
from pathlib import Path
from flit_core.common import Module
flit_module = Module("flit_core", Path(flit_core.__file__).parent.parent)
print(f"Module class available: {Module is not None}")
assert flit_module.is_package and flit_module.path == Path(flit_core.__file__).parent, (flit_module.is_package, flit_module.path)

print(f"flit_core utilities explored successfully")
